In [ ]:
# Copyright 2023 The Cirq Developers
# SPDX-License-Identifier: Apache-2.0
# Adapted from Qualtran v0.7.0, revision
# bcb17bb25360ed68ce3bc084645db3e929799389.


# Named register gates

Define an operation in terms of ordered register ports, bind those ports to physical wires, and append it to a FlagQuantum circuit. This guide maps all 17 cells of the locked register-gate tutorial.


## Signatures

A register has a name, a type carrying its width, and an optional array shape. A signature preserves register order. Core owns this metadata, while Compiler owns its typed calls and physical binding.


In [ ]:
from flagquantum.core.ir.program.register_types import opaque_register_type
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature

control_reg = Register("control", opaque_register_type(2))
target_reg = Register("target", opaque_register_type(3))
control_reg, target_reg


In [ ]:
signature = Signature((control_reg, target_reg))
signature


The width-based factory constructs the same ordered signature.


In [ ]:
assert tuple(signature) == tuple(Signature.build(control=2, target=3))


## An operation with named ports

The reusable example stores the source's ordered control/target loop as six calls to one shared CX definition. Its signature remains native metadata. The existing physical binder supplies operands for `circuit.append(op)`.


In [ ]:
from examples.algorithms.register_gates import RegisterControlledTargets


In [ ]:
gate = RegisterControlledTargets(control=(0, 1), target=(2, 3, 4))
program = gate.program()


In [ ]:
assert gate.signature.n_qubits() == 5


The operation binds each named register to an ordered physical wire tuple. The binder verifies ownership and keeps the declared bit order. An optional SDK adapter owns external qubit names separately.


In [ ]:
wire_ports = {"control": gate.control, "target": gate.target}
assert wire_ports == {"control": (0, 1), "target": (2, 3, 4)}


In [ ]:
operation = gate
bound = operation.bind()
assert bound.outputs == (gate.control, gate.target)


In [ ]:
import flagquantum as fq

circuit = fq.Circuit(5)
circuit.append(operation)
diagram = circuit.draw()
print(diagram)


## Inspect the stored graph and emitted circuit

The shared graph and the emitted circuit describe distinct levels of the same operation. Split/join calls preserve typed register ownership and emit zero physical gates. The six emitted CX operations consume no workspace or measurements. Their source QEC estimate is six Cliffords.


In [ ]:
from collections import Counter

from flagquantum.drawer.component_graph import export_component_graph_dot

call_graph = export_component_graph_dot(program, typed=True)
observed_counts = Counter(gate.name for gate in bound.circuit.instructions)
assert observed_counts == {"cx": 6}
assert bound.events == ()
print("signature:", tuple(gate.signature))
print("wire ports:", wire_ports)
print("outputs:", bound.outputs)
print("emitted operations:", dict(observed_counts))
print("stored graph:\n", call_graph)
